# Sophia AI - Pipeline de Entrenamiento y Diagnostico de Negocios

Este notebook contiene el entorno de pruebas, reentrenamiento e inferencia local de **Sophia AI**. Está diseñado bajo buenas prácticas para ser modular, legible y completamente independiente de archivos de código externos.

In [ ]:
from google.colab import userdata
userdata.get('secretName')

SecretNotFoundError: Secret secretName does not exist.

## 1. Inicialización y Configuración de Rutas

In [ ]:
import os
import sys
import json
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# Configuración de rutas base de forma robusta
NOTEBOOK_DIR = Path(os.getcwd())
if NOTEBOOK_DIR.name == 'notebooks':
    ROOT_DIR = NOTEBOOK_DIR.parent
else:
    ROOT_DIR = NOTEBOOK_DIR
MODEL_PATH = ROOT_DIR / "models" / "modelo_sophia_final.pt"
JSON_PATH = ROOT_DIR / "src_py" / "data" / "productos_metadata.json"

# Soporte de codificación para diferentes entornos de ejecución
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# Detectar si se ejecuta en Google Colab para cargar secretos de forma segura
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ejecutando en Google Colab. Cargando secretos desde Colab Secrets (userdata)...")
    from google.colab import userdata
    for clave in ['DATABASE_URL', 'GEMINI_API_KEY', 'MODEL_UPLOAD_TOKEN', 'DEPLOYMENT_SERVER_URL']:
        try:
            valor = userdata.get(clave)
            if valor:
                os.environ[clave] = valor
        except Exception:
            pass
else:
    load_dotenv(dotenv_path=ROOT_DIR / '.env')

# Descarga automatica de archivos requeridos en Google Colab desde GitHub
if IN_COLAB:
    import urllib.request

    csv_target = ROOT_DIR / 'src_py' / 'data' / 'intermediate' / 'compras_ctx.csv'
    if not csv_target.exists():
        print('Descargando compras_ctx.csv de respaldo desde GitHub...')
        csv_target.parent.mkdir(parents=True, exist_ok=True)
        url_csv = 'https://raw.githubusercontent.com/LeoSanchez22/TallerIntegrador1_Sanchez-R_Sanchez-C/feature/RestriccionXAI/src_py/data/intermediate/compras_ctx.csv'
        try:
            urllib.request.urlretrieve(url_csv, csv_target)
            print('[OK] compras_ctx.csv descargado con exito.')
        except Exception as e:
            print(f'Error descargando csv: {e}')

    json_target = ROOT_DIR / 'src_py' / 'data' / 'productos_metadata.json'
    if not json_target.exists():
        print('Descargando productos_metadata.json desde GitHub...')
        json_target.parent.mkdir(parents=True, exist_ok=True)
        url_json = 'https://raw.githubusercontent.com/LeoSanchez22/TallerIntegrador1_Sanchez-R_Sanchez-C/feature/RestriccionXAI/src_py/data/productos_metadata.json'
        try:
            urllib.request.urlretrieve(url_json, json_target)
            print('[OK] productos_metadata.json descargado con exito.')
        except Exception as e:
            print(f'Error descargando json: {e}')
    model_target = ROOT_DIR / 'models' / 'modelo_sophia_final.pt'
    if not model_target.exists():
        print('Descargando modelo_sophia_final.pt de respaldo desde GitHub...')
        model_target.parent.mkdir(parents=True, exist_ok=True)
        url_model = 'https://github.com/LeoSanchez22/TallerIntegrador1_Sanchez-R_Sanchez-C/raw/feature/RestriccionXAI/models/modelo_sophia_final.pt'
        try:
            urllib.request.urlretrieve(url_model, model_target)
            print('[OK] modelo_sophia_final.pt descargado con exito.')
        except Exception as e:
            print(f'Error descargando modelo: {e}')

Ejecutando en Google Colab. Cargando secretos desde Colab Secrets (userdata)...
Descargando compras_ctx.csv de respaldo desde GitHub...
[OK] compras_ctx.csv descargado con exito.
Descargando productos_metadata.json desde GitHub...
[OK] productos_metadata.json descargado con exito.
Descargando modelo_sophia_final.pt de respaldo desde GitHub...
[OK] modelo_sophia_final.pt descargado con exito.


## 2. Carga de Datos (Supabase con fallback a CSV)

In [ ]:
def cargar_datos_ventas():
    db_uri = os.environ.get("DATABASE_URL")
    if db_uri and "tu_contraseña" not in db_uri:
        try:
            engine = create_engine(db_uri)
            compras = pd.read_sql_query("SELECT * FROM ventas_detalle", con=engine)
            print("Conexión exitosa: Datos cargados desde Supabase.")
            return compras
        except Exception as e:
            print(f"No se pudo conectar a Supabase ({e}). Buscando archivo local...")

    # Fallback local
    compras_path = ROOT_DIR / "src_py" / "data" / "intermediate" / "compras_ctx.csv"
    if not compras_path.exists():
        compras_path = ROOT_DIR / "data" / "intermediate" / "compras_ctx.csv"

    if compras_path.exists():
        print(f"Cargando archivo de respaldo local: {compras_path.name}")
        return pd.read_csv(compras_path)

    print("Error: No se encontró la base de datos ni el archivo CSV local.")
    return None

df_compras = cargar_datos_ventas()

Conexión exitosa: Datos cargados desde Supabase.


## 3. Base de Conocimiento Clínico y Motor de Contenido (TF-IDF)

#### 3.1. Definición del Catálogo y Texto Clínico

In [ ]:
CONOCIMIENTO_BASE = {
    "LAGRICEL": {
        "sub_familia":   "Lubricante ocular",
        "mecanismo":     "lubricante hidratante lagrimal hialuronato sodio viscosidad retención acuosa",
        "indicacion":    "ojo seco síndrome ojo seco irritación ocular sequedad lubricación",
        "composicion":   "hialuronato de sodio 0.15% solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "LAGRICEL PF": {
        "sub_familia":   "Lubricante ocular sin conservantes",
        "mecanismo":     "lubricante hidratante lagrimal hialuronato sodio sin conservantes preservante",
        "indicacion":    "ojo seco severo intolerancia conservantes uso frecuente lubricación",
        "composicion":   "hialuronato de sodio libre de conservadores monodosis",
        "formato":       "Gotas monodosis",
        "es_nuevo":      False,
    },
    "ELIPTIC": {
        "sub_familia":   "Lubricante ocular",
        "mecanismo":     "lubricante hidratante lagrimal carboximetilcelulosa estabilización película lagrimal",
        "indicacion":    "ojo seco irritación sequedad ocular lubricación protección corneal",
        "composicion":   "carboximetilcelulosa sódica solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "ELIPTIC PF": {
        "sub_familia":   "Lubricante ocular sin conservantes",
        "mecanismo":     "lubricante hidratante lagrimal carboximetilcelulosa sin conservantes preservante",
        "indicacion":    "ojo seco severo intolerancia conservantes lubricación protección corneal",
        "composicion":   "carboximetilcelulosa sódica libre de conservadores",
        "formato":       "Gotas monodosis",
        "es_nuevo":      False,
    },
    "GAAP": {
        "sub_familia":   "Lubricante ocular",
        "mecanismo":     "lubricante hidratante lagrimal polivinilpirrolidona glicol propileno viscosidad",
        "indicacion":    "ojo seco irritación sequedad ocular lubricación confort visual",
        "composicion":   "polivinilpirrolidona polietilenglicol solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "GAAP PF": {
        "sub_familia":   "Lubricante ocular sin conservantes",
        "mecanismo":     "lubricante hidratante lagrimal polivinilpirrolidona sin conservantes preservante",
        "indicacion":    "ojo seco severo intolerancia conservantes lubricación confort visual",
        "composicion":   "polivinilpirrolidona polietilenglicol libre de conservadores",
        "formato":       "Gotas monodosis",
        "es_nuevo":      False,
    },
    "AQUADRAN": {
        "sub_familia":   "Lubricante ocular gel",
        "mecanismo":     "lubricante hidratante gel carbómero retención prolongada película lagrimal nocturno",
        "indicacion":    "ojo seco severo nocturno sequedad extrema protección corneal gel",
        "composicion":   "carbómero gel oftálmico",
        "formato":       "Gel",
        "es_nuevo":      False,
    },
    "DUSTALOX": {
        "sub_familia":   "Antibiótico antiinflamatorio ocular",
        "mecanismo":     "corticoide antibiótico dexametasona tobramicina antiinflamatorio antibacteriano",
        "indicacion":    "inflamación ocular infección bacteriana blefaritis conjuntivitis bacteriana postquirúrgico",
        "composicion":   "dexametasona 0.1% tobramicina 0.3% suspensión oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "FLUMETOL NF": {
        "sub_familia":   "Corticoide ocular",
        "mecanismo":     "corticoide fluorometolona antiinflamatorio esteroide ocular",
        "indicacion":    "inflamación ocular no infecciosa queratoconjuntivitis alergica postquirúrgico uveítis anterior",
        "composicion":   "fluorometolona 0.1% suspensión oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "SOPHIPREN": {
        "sub_familia":   "Corticoide ocular",
        "mecanismo":     "corticoide prednisolona antiinflamatorio esteroide potente ocular",
        "indicacion":    "inflamación ocular severa uveítis postquirúrgico queratitis corticoide",
        "composicion":   "prednisolona acetato 1% suspensión oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "ELAR-B": {
        "sub_familia":   "Corticoide antibiótico ocular",
        "mecanismo":     "corticoide antibiótico betametasona antiinflamatorio antibacteriano combinado",
        "indicacion":    "inflamación ocular infección bacteriana combinada blefaroconjuntivitis",
        "composicion":   "betametasona neomicina solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "TRAZIDEX O": {
        "sub_familia":   "Antibiótico antiinflamatorio ocular",
        "mecanismo":     "antibiótico antiinflamatorio tramadol dexametasona bactericida ocular gotas",
        "indicacion":    "conjuntivitis bacteriana blefaritis infección ocular postquirúrgico antibacteriano",
        "composicion":   "tobramicina dexametasona solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "TRAZIDEX U": {
        "sub_familia":   "Antibiótico antiinflamatorio ungüento",
        "mecanismo":     "antibiótico antiinflamatorio tobramicina dexametasona ungüento bactericida ocular nocturno",
        "indicacion":    "conjuntivitis bacteriana blefaritis infección ocular ungüento nocturno",
        "composicion":   "tobramicina dexametasona ungüento oftálmico",
        "formato":       "Ungüento",
        "es_nuevo":      False,
    },
    "ZEBESTEN": {
        "sub_familia":   "Antihistamínico antialérgico ocular",
        "mecanismo":     "antihistamínico estabilizador mastocitos ketotifeno antialérgico antipruriginoso",
        "indicacion":    "alergia ocular conjuntivitis alérgica prurito ocular picazón lagrimeo alérgico",
        "composicion":   "ketotifeno 0.025% solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "AGGLAD": {
        "sub_familia":   "Antihistamínico antialérgico ocular",
        "mecanismo":     "antihistamínico antialérgico olopatadina bloqueador H1 mastocitos estabilizador",
        "indicacion":    "conjuntivitis alérgica alergia ocular prurito picazón lagrimeo rojez alérgica",
        "composicion":   "olopatadina 0.1% solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    },
    "LANDAX": {
        "sub_familia":   "Antihistamínico antialérgico ocular",
        "mecanismo":     "antihistamínico antialérgico epinastina bloqueador H1 estabilizador mastocitos",
        "indicacion":    "conjuntivitis alérgica alergia ocular prurito picazón lagrimeo estacional",
        "composicion":   "epinastina 0.05% solución oftálmica",
        "formato":       "Gotas",
        "es_nuevo":      False,
    }
}

def cargar_metadatos_nuevos(ruta: Path) -> dict:
    if ruta.exists():
        try:
            with open(ruta, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def _texto_clinico(meta: dict) -> str:
    partes = [
        meta.get("sub_familia", ""),
        meta.get("mecanismo", ""),
        meta.get("indicacion", ""),
        meta.get("composicion", ""),
        meta.get("formato", ""),
    ]
    return " ".join(p for p in partes if p).lower()

#### 3.2. Implementación del Motor de Similitud por Contenido

In [ ]:
class MotorContenido:
    UMBRAL = 0.05
    def __init__(self, df_compras: pd.DataFrame = None, ruta_json: Path = JSON_PATH):
        registro = {k: dict(v) for k, v in CONOCIMIENTO_BASE.items()}
        nuevos = cargar_metadatos_nuevos(ruta_json)
        for clave, meta in nuevos.items():
            registro[clave] = {**meta, "es_nuevo": True}
        self.registro = registro
        self.productos = list(registro.keys())
        corpus = [_texto_clinico(registro[p]) for p in self.productos]
        self._vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95, sublinear_tf=True)
        self._tfidf_matrix = self._vectorizer.fit_transform(corpus)
        self.p2i = {p: i for i, p in enumerate(self.productos)}

    def hay_productos_nuevos(self) -> bool:
        return any(m.get("es_nuevo") for m in self.registro.values())

    def productos_nuevos(self) -> list:
        return [p for p, m in self.registro.items() if m.get("es_nuevo")]

    def similitud_par(self, prod_a: str, prod_b: str) -> float:
        ka, kb = prod_a.upper().strip(), prod_b.upper().strip()
        if ka not in self.p2i or kb not in self.p2i:
            return 0.0
        ia, ib = self.p2i[ka], self.p2i[kb]
        sim = cosine_similarity(self._tfidf_matrix[ia], self._tfidf_matrix[ib])[0][0]
        return round(float(sim), 4)

    def tabla_similitud_producto_nuevo(self, nombre_nuevo: str) -> pd.DataFrame:
        clave = nombre_nuevo.upper().strip()
        if clave not in self.p2i:
            return pd.DataFrame()
        filas = []
        for prod, meta in self.registro.items():
            if prod == clave or meta.get("es_nuevo"):
                continue
            sim = self.similitud_par(clave, prod)
            meta_nuevo = self.registro[clave]
            coincidencias = []
            if meta.get("sub_familia") == meta_nuevo.get("sub_familia"):
                coincidencias.append(f"misma sub-familia ({meta['sub_familia']})")
            if meta.get("formato") == meta_nuevo.get("formato"):
                coincidencias.append(f"mismo formato ({meta['formato']})")
            palabras_nuevo = set(_texto_clinico(meta_nuevo).split())
            palabras_prod  = set(_texto_clinico(meta).split())
            comunes = palabras_nuevo & palabras_prod - {"de", "y", "el", "la", "en", "0.1%", "5%"}
            if comunes:
                top_comunes = sorted(comunes, key=len, reverse=True)[:3]
                coincidencias.append(f"comparten: {', '.join(top_comunes)}")
            filas.append({
                "Producto existente": prod,
                "Sub-familia":        meta.get("sub_familia", ""),
                "Similitud TF-IDF":   f"{sim*100:.1f}%",
                "¿Por qué?":          " | ".join(coincidencias) if coincidencias else "perfil diferente",
                "_score_raw":         sim,
            })
        df = pd.DataFrame(filas).sort_values("_score_raw", ascending=False).drop(columns=["_score_raw"])
        return df

    def score(self, historial_nombres: list, producto_nuevo: str) -> tuple:
        clave = producto_nuevo.upper().strip()
        if clave not in self.p2i:
            return 0.0, "", ""
        mejor_score, mejor_prod, mejor_razon = 0.0, "", ""
        for prod in historial_nombres:
            sim = self.similitud_par(clave, prod)
            if sim > mejor_score:
                mejor_score = sim
                mejor_prod  = prod
                meta_nuevo = self.registro.get(clave, {})
                meta_hist  = self.registro.get(prod.upper().strip(), {})
                partes_razon = []
                if meta_hist.get("sub_familia") == meta_nuevo.get("sub_familia"):
                    partes_razon.append(f"misma sub-familia terapéutica ({meta_nuevo.get('sub_familia')})")
                if meta_hist.get("formato") == meta_nuevo.get("formato"):
                    partes_razon.append("mismo formato de administración")
                palabras_n = set(_texto_clinico(meta_nuevo).split())
                palabras_h = set(_texto_clinico(meta_hist).split())
                comunes = palabras_n & palabras_h - {"de", "y", "el", "la", "en"}
                if comunes:
                    top = sorted(comunes, key=len, reverse=True)[:2]
                    partes_razon.append(f"principios activos relacionados: {', '.join(top)}")
                mejor_razon = "; ".join(partes_razon) if partes_razon else "afinidad de perfil terapéutico"
        return round(mejor_score, 4), mejor_prod, mejor_razon

    def recomendar_nuevos(self, historial_nombres: list, top_k: int = 3) -> list:
        historial_upper = {h.upper().strip() for h in historial_nombres}
        resultados = []
        for prod_nuevo in self.productos_nuevos():
            if prod_nuevo in historial_upper:
                continue
            s, match, razon = self.score(historial_nombres, prod_nuevo)
            if s >= self.UMBRAL:
                meta = self.registro[prod_nuevo]
                resultados.append({
                    "producto":    prod_nuevo,
                    "score":       s,
                    "similar_a":   match,
                    "razon_xai":   razon,
                    "sub_familia": meta.get("sub_familia", ""),
                    "indicacion":  meta.get("indicacion", ""),
                    "composicion": meta.get("composicion", ""),
                    "motor":       "Contenido (Nuevo Lanzamiento)",
                    "es_nuevo":    True,
                })
        resultados.sort(key=lambda x: x["score"], reverse=True)
        return resultados[:top_k]

def inyectar_candidatos_nuevos(motor: MotorContenido, historial_nombres: list, mapa_nombre_a_id: dict) -> list:
    nuevos = motor.recomendar_nuevos(historial_nombres)
    candidatos = []
    for rec in nuevos:
        nombre  = rec["producto"]
        prod_id = mapa_nombre_a_id.get(nombre, f"NUEVO_{nombre}")
        candidatos.append((prod_id, rec["motor"], rec["score"], rec))
    return candidatos

## 4. Arquitectura y Dataset de Deep Learning (PyTorch)

#### 4.1. Arquitectura del Modelo AttentionGRUMejorado

In [ ]:
class AttentionGRUMejorado(nn.Module):
    def __init__(self, num_items, num_clientes, num_meses=13,
                 embedding_dim=64, hidden_dim=128, dropout=0.3):
        super(AttentionGRUMejorado, self).__init__()

        self.item_embedding    = nn.Embedding(num_items,    embedding_dim, padding_idx=0)
        self.cliente_embedding = nn.Embedding(num_clientes, embedding_dim // 2)
        self.mes_embedding     = nn.Embedding(num_meses,    embedding_dim // 4)

        gru_input_dim = embedding_dim + embedding_dim // 2 + embedding_dim // 4

        self.gru             = nn.GRU(gru_input_dim, hidden_dim, batch_first=True, num_layers=2, dropout=dropout)
        self.attention_layer = nn.Linear(hidden_dim, 1)
        self.dropout         = nn.Dropout(dropout)
        self.fc              = nn.Linear(hidden_dim, num_items)

    def forward(self, seq_items, cliente_id, mes_id):
        B, L = seq_items.shape
        item_emb = self.item_embedding(seq_items)

        cli_emb = self.cliente_embedding(cliente_id).unsqueeze(1).expand(B, L, -1)
        mes_emb = self.mes_embedding(mes_id).unsqueeze(1).expand(B, L, -1)

        x = torch.cat([item_emb, cli_emb, mes_emb], dim=-1)
        x = self.dropout(x)

        gru_out, _ = self.gru(x)

        attn_scores   = self.attention_layer(gru_out)
        attn_weights  = F.softmax(attn_scores, dim=1)
        context       = torch.sum(attn_weights * gru_out, dim=1)
        context       = self.dropout(context)

        logits = self.fc(context)
        return logits, attn_weights


#### 4.2. Dataset SophiaDataset con Sliding Window

In [ ]:
class SophiaDataset(Dataset):
    def __init__(self, registros, seq_len, cliente2idx):
        self.muestras = []
        self.seq_len  = seq_len
        for (cliente_id_orig, mes), grupo in registros:
            cli_idx = cliente2idx.get(cliente_id_orig, 0)
            mes_idx = int(mes) if mes is not None else 1
            ids     = grupo['producto_id'].tolist()
            for fin in range(len(ids) - 1, 0, -1):
                target  = ids[fin]
                inicio  = max(0, fin - seq_len)
                ctx     = ids[inicio:fin]
                pad_len = seq_len - len(ctx)
                ctx_pad = [0] * pad_len + ctx
                self.muestras.append({
                    "seq":       torch.tensor(ctx_pad, dtype=torch.long),
                    "cliente":   torch.tensor(cli_idx, dtype=torch.long),
                    "mes":       torch.tensor(mes_idx, dtype=torch.long),
                    "target":    torch.tensor(target,  dtype=torch.long),
                })

    def __len__(self):
        return len(self.muestras)

    def __getitem__(self, idx):
        return self.muestras[idx]

#### 4.3. Métricas de Validación MLOps

In [ ]:
def hit_rate_at_k(logits_batch, targets, k=5):
    topk = torch.topk(logits_batch, k, dim=1).indices
    hits = (topk == targets.unsqueeze(1)).any(dim=1).float()
    return hits.mean().item()

def ndcg_at_k(logits_batch, targets, k=5):
    topk     = torch.topk(logits_batch, k, dim=1).indices
    ndcg_sum = 0.0
    for i, target in enumerate(targets):
        match = (topk[i] == target).nonzero(as_tuple=True)
        if len(match[0]) > 0:
            pos = match[0][0].item()
            ndcg_sum += 1.0 / np.log2(pos + 2)
    return ndcg_sum / len(targets)

## 5. Preparación de Datos y Bucle de Entrenamiento

#### 5.1. Preparación de Datos de Entrenamiento

In [ ]:
def preparar_datos_entrenamiento(compras, config):
    df = compras.copy()
    try:
        df['cliente_id'] = df['cliente_id'].astype(int)
        df['producto_id'] = df['producto_id'].astype(int)
        df['cantidad'] = df['cantidad'].fillna(0).astype(int)
    except Exception as e:
        print(f"Advertencia en formateo: {e}")

    col_cliente_id = 'cliente_id'

    # Resolver estacionalidad
    mes_disponible = False
    for col_fecha in ['fecha', 'date', 'fecha_pedido']:
        if col_fecha in df.columns:
            df['mes'] = pd.to_datetime(df[col_fecha], errors='coerce').dt.month.fillna(1).astype(int)
            mes_disponible = True
            break
    if not mes_disponible:
        df['mes'] = 1

    # Filtrar secuencias mínimas
    conteos = df.groupby(col_cliente_id)['producto_id'].count()
    clientes_validos = conteos[conteos >= config["min_seq_len"]].index
    df = df[df[col_cliente_id].isin(clientes_validos)].copy()

    clientes_unicos = sorted(df[col_cliente_id].unique().tolist())
    cliente2idx     = {int(c): i + 1 for i, c in enumerate(clientes_unicos)}
    num_clientes    = len(clientes_unicos) + 1
    num_items       = int(df['producto_id'].max()) + 2

    # División Train/Val/Test
    np.random.seed(config["seed"])
    np.random.shuffle(clientes_unicos)
    n_val  = max(1, int(len(clientes_unicos) * config["val_split"]))
    n_test = max(1, int(len(clientes_unicos) * config["test_split"]))

    clientes_test  = set(clientes_unicos[:n_test])
    clientes_val   = set(clientes_unicos[n_test:n_test + n_val])
    clientes_train = set(clientes_unicos[n_test + n_val:])

    def filtrar_y_agrupar(d_frame, clientes_set):
        subset = d_frame[d_frame[col_cliente_id].isin(clientes_set)]
        return subset.groupby([col_cliente_id, 'mes'])

    grupos_train = filtrar_y_agrupar(df, clientes_train)
    grupos_val   = filtrar_y_agrupar(df, clientes_val)
    grupos_test  = filtrar_y_agrupar(df, clientes_test)

    ds_train = SophiaDataset(grupos_train, config["seq_len"], cliente2idx)
    ds_val   = SophiaDataset(grupos_val,   config["seq_len"], cliente2idx)
    ds_test  = SophiaDataset(grupos_test,  config["seq_len"], cliente2idx)

    return ds_train, ds_val, ds_test, num_items, num_clientes, cliente2idx

#### 5.2. Función del Bucle de Entrenamiento

In [ ]:
def subir_modelo_a_despliegue(ruta_modelo, url_servidor, token):
    if not token or not url_servidor:
        print("Sincronizacion omitida: Falta DEPLOYMENT_SERVER_URL o MODEL_UPLOAD_TOKEN en las variables de entorno o Colab Secrets.")
        return

    import requests
    url = f"{url_servidor.rstrip('/')}/model/upload"
    headers = {"Authorization": f"Bearer {token}"}

    print(f"Sincronizando checkpoint '{ruta_modelo.name}' con el despliegue cloud ({url})...")
    try:
        with open(ruta_modelo, "rb") as f:
            files = {"file": (ruta_modelo.name, f, "application/octet-stream")}
            response = requests.post(url, headers=headers, files=files, timeout=60)

        if response.status_code == 200:
            print("[OK] Checkpoint sincronizado exitosamente con el despliegue cloud.")
        else:
            print(f"[ERROR] Error de sincronizacion ({response.status_code}): {response.text}")
    except Exception as e:
        print(f"[ERROR] Fallo la conexion con el servidor de despliegue: {e}")

def entrenar_modelo_local(compras, config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Entrenando en: {device}")

    Path(config["output_model"]).parent.mkdir(parents=True, exist_ok=True)
    ds_train, ds_val, ds_test, num_items, num_clientes, cliente2idx = preparar_datos_entrenamiento(compras, config)

    dl_train = DataLoader(ds_train, batch_size=config["batch_size"], shuffle=True, drop_last=True)
    dl_val   = DataLoader(ds_val,   batch_size=config["batch_size"], shuffle=False, drop_last=False)

    modelo = AttentionGRUMejorado(
        num_items    = num_items,
        num_clientes = num_clientes,
        embedding_dim = config["embedding_dim"],
        hidden_dim    = config["hidden_dim"],
        dropout       = config["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(modelo.parameters(), lr=config["lr"], weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    criterion = nn.CrossEntropyLoss(ignore_index=0)

    mejor_hr_val = 0.0
    epocas_sin_mejora = 0

    for epoch in range(1, config["epochs"] + 1):
        modelo.train()
        loss_total = 0.0
        for batch in dl_train:
            seq     = batch["seq"].to(device)
            cliente = batch["cliente"].to(device)
            mes     = batch["mes"].to(device)
            target  = batch["target"].to(device)

            optimizer.zero_grad()
            logits, _ = modelo(seq, cliente, mes)
            loss = criterion(logits, target)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            optimizer.step()
            loss_total += loss.item()

        loss_media = loss_total / len(dl_train)

        modelo.eval()
        hr_val, ndcg_val = 0.0, 0.0
        n_batches_val = 0
        with torch.no_grad():
            for batch in dl_val:
                seq     = batch["seq"].to(device)
                cliente = batch["cliente"].to(device)
                mes     = batch["mes"].to(device)
                target  = batch["target"].to(device)
                logits, _ = modelo(seq, cliente, mes)
                hr_val   += hit_rate_at_k(logits, target, k=config["k_eval"])
                ndcg_val += ndcg_at_k(logits, target, k=config["k_eval"])
                n_batches_val += 1

        hr_val /= n_batches_val
        ndcg_val /= n_batches_val
        scheduler.step(1 - hr_val)

        print(f"Época {epoch:>2}/{config['epochs']} | Loss: {loss_media:.4f} | HR@5 Val: {hr_val*100:.1f}% | NDCG@5 Val: {ndcg_val:.4f}")

        if hr_val > mejor_hr_val:
            mejor_hr_val = hr_val
            epocas_sin_mejora = 0
            torch.save({
                "epoch":        epoch,
                "model_state":  modelo.state_dict(),
                "num_items":    num_items,
                "num_clientes": num_clientes,
                "cliente2idx":  cliente2idx,
                "config":       config,
                "hr_val":       hr_val,
                "ndcg_val":     ndcg_val,
            }, config["output_model"])
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= config["early_stop_patience"]:
                print(f"Early stopping en época {epoch}.")
                break

    print("Proceso de entrenamiento finalizado. Checkpoint guardado.")
    # Sincronización automática con el servidor de despliegue
    ruta_modelo = Path(config["output_model"])
    url_servidor = os.environ.get("DEPLOYMENT_SERVER_URL")
    token = os.environ.get("MODEL_UPLOAD_TOKEN")
    subir_modelo_a_despliegue(ruta_modelo, url_servidor, token)

#### 5.3. Ejecución del Entrenamiento

In [ ]:
CONFIG_ENTRENAMIENTO = {
    "output_model":    MODEL_PATH,
    "embedding_dim":   64,
    "hidden_dim":      128,
    "dropout":         0.3,
    "lr":              0.001,
    "epochs":          50,
    "batch_size":      64,
    "seq_len":         10,
    "k_eval":          5,
    "early_stop_patience": 8,
    "val_split":       0.10,
    "test_split":      0.05,
    "min_seq_len":     3,
    "seed":            42,
}

entrenar_modelo_local(df_compras, CONFIG_ENTRENAMIENTO)

Entrenando en: cpu
Época  1/50 | Loss: 2.6080 | HR@5 Val: 48.6% | NDCG@5 Val: 0.2963
Época  2/50 | Loss: 2.5010 | HR@5 Val: 52.2% | NDCG@5 Val: 0.3280
Época  3/50 | Loss: 2.4289 | HR@5 Val: 55.9% | NDCG@5 Val: 0.3432
Época  4/50 | Loss: 2.3610 | HR@5 Val: 55.3% | NDCG@5 Val: 0.3487
Época  5/50 | Loss: 2.2104 | HR@5 Val: 66.2% | NDCG@5 Val: 0.4546
Época  6/50 | Loss: 1.9780 | HR@5 Val: 73.1% | NDCG@5 Val: 0.5346
Época  7/50 | Loss: 1.7972 | HR@5 Val: 74.9% | NDCG@5 Val: 0.5543
Época  8/50 | Loss: 1.6691 | HR@5 Val: 76.5% | NDCG@5 Val: 0.5781
Época  9/50 | Loss: 1.5623 | HR@5 Val: 77.8% | NDCG@5 Val: 0.5785
Época 10/50 | Loss: 1.4961 | HR@5 Val: 78.3% | NDCG@5 Val: 0.5856
Época 11/50 | Loss: 1.4359 | HR@5 Val: 76.7% | NDCG@5 Val: 0.5792
Época 12/50 | Loss: 1.3698 | HR@5 Val: 78.3% | NDCG@5 Val: 0.5760
Época 13/50 | Loss: 1.3325 | HR@5 Val: 78.7% | NDCG@5 Val: 0.5866
Época 14/50 | Loss: 1.2798 | HR@5 Val: 78.3% | NDCG@5 Val: 0.5913
Época 15/50 | Loss: 1.2328 | HR@5 Val: 78.7% | NDCG@5 Val

## 6. Selección del Cliente de Prueba

In [ ]:
CLIENTE_ID_PRUEBA = 27

col_cliente_id = 'cliente_id'
col_cliente_nom = next((c for c in ['cliente', 'nombre_cliente', 'Cliente'] if c in df_compras.columns), 'cliente')

df_filtro = df_compras[df_compras[col_cliente_id] == CLIENTE_ID_PRUEBA]
if df_filtro.empty:
    CLIENTE_ID_PRUEBA = int(df_compras[col_cliente_id].iloc[0])
    nombre_cliente = df_compras[col_cliente_nom].iloc[0]
    print(f"Cliente fallback seleccionado: ID {CLIENTE_ID_PRUEBA} ({nombre_cliente})")
else:
    nombre_cliente = df_filtro[col_cliente_nom].iloc[0]
    print(f"Cliente de prueba activo: ID {CLIENTE_ID_PRUEBA} ({nombre_cliente})")

Cliente de prueba activo: ID 27 (BOTICA IBIS S.A.C.)


## 7. Lectura y Telemetría Global del Checkpoint

In [ ]:
try:
    if MODEL_PATH.exists():
        checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
        hr_val = float(checkpoint.get('hr_val', 0.793))
        ndcg_val = float(checkpoint.get('ndcg_val', 0.605))
        epoch = int(checkpoint.get('epoch', 25))

        print(f"Estado del modelo:        Healthy")
        print(f"Época de guardado:        {epoch}")
        print(f"Precisión@5 (Hit Rate):   {hr_val * 100:.2f}%")
        print(f"NDCG@5 (Ranking):         {ndcg_val:.4f}")
    else:
        print("Advertencia: No se encontró el checkpoint. Usando valores base (Hit Rate: 79.3%, NDCG: 0.6049).")
except Exception as e:
    print(f"Error al leer el checkpoint: {e}")

Estado del modelo:        Healthy
Época de guardado:        18
Precisión@5 (Hit Rate):   78.97%
NDCG@5 (Ranking):         0.5947


## 8. Historial Transaccional del Comprador

In [ ]:
historial_cliente = df_compras[df_compras[col_cliente_id] == CLIENTE_ID_PRUEBA].copy()
col_mes = next((c for c in ['mes_nombre', 'mes_abbr', 'mes_num'] if c in historial_cliente.columns), None)
col_cant = next((c for c in ['cantidad', 'Cantidad'] if c in historial_cliente.columns), None)

columnas_historico = ['producto']
if col_mes: columnas_historico.append(col_mes)
if col_cant: columnas_historico.append(col_cant)
if 'monto_cancelado' in historial_cliente.columns: columnas_historico.append('monto_cancelado')

display(historial_cliente[columnas_historico].head(15))

,producto,mes_nombre,cantidad,monto_cancelado
1175,AGGLAD,ABRIL,5,275.0
1176,AQUADRAN,ABRIL,6,324.0
1177,DUSTALOX,ABRIL,6,271.0
1178,ELAR-B,ABRIL,6,345.0
1179,ELIPTIC PF,ABRIL,10,372.0
1180,FLUMETOL NF,ABRIL,5,225.0
1181,GAAP PF,ABRIL,10,559.0
1182,LAGRICEL PF,ABRIL,6,286.0
1183,SOPHIPREN,ABRIL,6,280.0
1184,ZEBESTEN,ABRIL,5,245.0


## 9. Reglas de Negocio de Sophia

#### 9.1. Filtros de Seguridad y Control de Stock Zonal

In [ ]:
def obtener_quiebres_zona(zona, compras_df):
    col_zona = next((c for c in ['vendedor', 'zona'] if c in compras_df.columns), 'zona')
    if 'sin_stock' in compras_df.columns:
        return compras_df[(compras_df[col_zona] == zona) & (compras_df['sin_stock'] == True)]['producto'].unique().tolist()
    elif 'stock' in compras_df.columns:
        return compras_df[(compras_df[col_zona] == zona) & (compras_df['stock'] == 0)]['producto'].unique().tolist()
    return []

def obtener_pares_canibalizacion(mapa_productos):
    pares = []
    nombres = list(mapa_productos.values())
    for nombre in nombres:
        base = nombre.replace(' PF', '').replace(' PLUS', '').strip()
        if base != nombre and base in nombres:
            pares.append((base, nombre))
    return pares

def pasa_filtros_seguridad(producto_sugerido, historial_cliente, zona_actual, compras_df, mapa_productos):
    quiebres = obtener_quiebres_zona(zona_actual, compras_df)
    if producto_sugerido in quiebres:
        return False, f"Sin stock en {zona_actual}."
    pares = obtener_pares_canibalizacion(mapa_productos)
    for base, premium in pares:
        if producto_sugerido == premium and base in historial_cliente:
            return False, f"Riesgo de canibalización: cliente ya consume {base}."
    return True, "Aprobado"


#### 9.2. Generador de Explicaciones Clínicas (Gemini XAI con Fallback Local)

In [ ]:
def generar_explicacion(producto_sugerido, historial_cliente, motor_origen, horizonte_mes, item_foco=None, peso=None, es_autorregresivo=False):
    api_key = os.environ.get("GEMINI_API_KEY")
    if api_key:
        try:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            mes_texto = horizonte_mes.split(" (")[1].replace(")", "").lower() if "(" in horizonte_mes else horizonte_mes.lower()
            historial_base = item_foco if motor_origen == 'Atención-GRU' else (historial_cliente[-1] if historial_cliente else "Productos habituales")

            logica_xai = ""
            if motor_origen == 'Atención-GRU':
                logica_xai = f"El modelo detectó una 'Causalidad GRU' secuencial. La capa de atención asignó {peso}% de relevancia al consumo histórico de '{historial_base}'. Esto indica un ciclo de reposición inminente en el tiempo."
            elif motor_origen == 'Cold Start':
                logica_xai = f"Activación de 'Cold Start'. Al carecer de historial suficiente, '{producto_sugerido}' se recomienda por tener alta adopción en otras clínicas de la zona."
            elif motor_origen == 'Contenido (Nuevo Lanzamiento)':
                logica_xai = f"El Motor de Similitud por Contenido calculó afinidad terapéutica entre '{producto_sugerido}' y '{historial_base}'. Comparten familia terapéutica y vía de administración. Este es un producto de nuevo lanzamiento sin historial de ventas: la recomendación se basa en la compatibilidad clínica de sus metadatos."
            else:
                logica_xai = f"El modelo detectó una 'Afinidad NCF'. Evaluando el Espacio Latente, encontró que clínicas con un perfil estructural idéntico a esta, que ya consumen '{historial_base}', tienen una probabilidad muy alta de adoptar '{producto_sugerido}'."

            if es_autorregresivo:
                logica_xai += f" IMPORTANTE: Esta es una proyección autorregresiva para el {mes_texto}. El sistema está asumiendo que las ventas sugeridas en los meses previos fueron cerradas con éxito, lo que obliga al algoritmo a mutar su sugerencia hacia una estrategia de expansión de catálogo."

            prompt = f"""
            Eres el motor de IA Explicable (XAI) de Laboratorios Sophia.
            Tu tarea es traducir la lógica matemática de nuestros modelos en una justificación clínica y comercial de EXACTAMENTE 2 a 3 líneas para el visitador médico.
            Variables:
            - Producto Sugerido: '{producto_sugerido}'
            - Detonante Histórico: '{historial_base}'
            - Proyección para: {mes_texto}
            - Razón Matemática: {logica_xai}
            Restricciones: Sin emojis, tono técnico y comercial persuasivo.
            """
            model = genai.GenerativeModel('gemini-1.5-flash', generation_config={"temperature": 0.6})
            respuesta = model.generate_content(prompt)
            return respuesta.text.strip().replace("🤖", "").replace("💊", "").replace("🚀", "").replace("💡", "")
        except Exception:
            pass

    suffix = " (proyección autorregresiva)" if es_autorregresivo else ""
    if motor_origen == 'Atención-GRU':
        return f"Reposición Sugerida: Ciclo de compra detecta demanda inminente para {producto_sugerido} basado en consumo de {item_foco} ({peso}% relevancia){suffix}."
    elif motor_origen == 'Cold Start':
        return f"Exito Local: {producto_sugerido} es uno de los productos mas solicitados en tu zona comercial{suffix}."
    elif motor_origen == 'Contenido (Nuevo Lanzamiento)':
        return f"Nuevo Lanzamiento: Recomendado por afinidad terapeutica de ingredientes activos con {item_foco}{suffix}."
    else:
        return f"Oportunidad Cross-Selling: Clinicas con perfil de compra similar al tuyo que adquieren {item_foco} tambien consumen {producto_sugerido}{suffix}."


## 10. Inferencia y Telemetría Zonal del Cliente

#### 10.1. Cargador de Modelos PyTorch desde Checkpoint

In [ ]:
def cargar_modelo_gru_checkpoint(model_path, num_items_fallback, num_clientes_fallback, cliente2idx_fallback):
    modelo_cargado = False
    cliente2idx = {}
    num_clientes = 0
    modelo_gru = None

    if model_path.exists():
        try:
            checkpoint = torch.load(model_path, map_location='cpu', weights_only=False)
            num_items_m = checkpoint['num_items']
            num_clientes = checkpoint['num_clientes']
            cliente2idx = checkpoint['cliente2idx']
            cfg = checkpoint.get('config', {})

            modelo_gru = AttentionGRUMejorado(
                num_items = num_items_m,
                num_clientes = num_clientes,
                embedding_dim = cfg.get('embedding_dim', 64),
                hidden_dim = cfg.get('hidden_dim', 128),
                dropout = 0.0,
            )
            modelo_gru.load_state_dict(checkpoint['model_state'])
            modelo_gru.eval()
            modelo_cargado = True
        except Exception as e:
            sys.stderr.write(f"[WARNING] Error modelo PyTorch al cargar checkpoint: {e}\n")

    if not modelo_cargado:
        cliente2idx = cliente2idx_fallback
        num_clientes = num_clientes_fallback
        modelo_gru = AttentionGRUMejorado(num_items=num_items_fallback, num_clientes=num_clientes, dropout=0.0)
        modelo_gru.eval()

    return modelo_gru, cliente2idx, num_clientes


#### 10.2. Recomendador Colaborativo Basado en Clientes Gemelos (NCF)

In [ ]:
def obtener_candidatos_ncf(cliente_id, cli_idx_tensor, modelo_gru, cliente2idx, compras_ctx, num_items, mapa_productos):
    cliente_adn = modelo_gru.cliente_embedding.weight[cli_idx_tensor[0]]
    todos_clientes_adn = modelo_gru.cliente_embedding.weight
    similitudes = F.cosine_similarity(cliente_adn.unsqueeze(0), todos_clientes_adn)
    similitudes[cli_idx_tensor[0]] = -1.0
    top_gemelos = torch.topk(similitudes, k=5)
    indices_gemelos = top_gemelos.indices.detach().numpy()

    ids_gemelos_reales = [k for k, v in cliente2idx.items() if v in indices_gemelos]
    compras_gemelos = compras_ctx[compras_ctx['cliente_id'].isin(ids_gemelos_reales)]
    frecuencia_gemelos = compras_gemelos['producto_id'].value_counts()
    scores_ncf = np.zeros(num_items)
    max_frecuencia_gemelos = frecuencia_gemelos.max() if not frecuencia_gemelos.empty else 1

    for pid, count in frecuencia_gemelos.items():
        if pid < num_items:
            scores_ncf[pid] = (count / max_frecuencia_gemelos) * 0.90

    top_indices_ncf = scores_ncf.argsort()[::-1][:5]
    candidatos_ncf = [(int(i + 1), 'NCF', scores_ncf[i]) for i in top_indices_ncf if (i + 1) in mapa_productos]
    return candidatos_ncf


#### 10.3. Evaluación de Desempeño Zonal (Telemetría Local)

In [ ]:
def calcular_telemetria_zonal(cliente_id, zona_activa, compras_ctx, modelo_gru, cliente2idx, mapa_productos, num_items):
    try:
        col_zona = next((c for c in ['vendedor', 'zona'] if c in compras_ctx.columns), 'zona')
        clientes_zona = compras_ctx[compras_ctx[col_zona] == zona_activa]['cliente_id'].drop_duplicates().tolist()
        semilla_dinamica = int(cliente_id) + len(zona_activa)
        np.random.seed(semilla_dinamica)
        tamanho_muestra = min(20, len(clientes_zona))
        clientes_muestra = np.random.choice(clientes_zona, tamanho_muestra, replace=False)

        hr_total, ndcg_total, atencion_media = 0.0, 0.0, 0.0
        productos_sugeridos_unicos = set()
        casos_validos = 0
        mes_actual = int(pd.Timestamp.now().month)

        for cid in clientes_muestra:
            hist_ids = compras_ctx[compras_ctx['cliente_id'] == int(cid)]['producto_id'].tolist()
            if len(hist_ids) < 3: continue
            contexto_ids = hist_ids[:-1]
            ground_truth_id = hist_ids[-1]
            contexto_nombres = [mapa_productos[pid] for pid in contexto_ids if pid in mapa_productos]
            ctx_ids_eval = contexto_ids[-10:]
            max_emb_id_eval = modelo_gru.item_embedding.num_embeddings - 1
            ctx_ids_seguros_eval = [pid if pid <= max_emb_id_eval else 0 for pid in ctx_ids_eval]
            pad_len_eval = max(0, 10 - len(ctx_ids_seguros_eval))
            ctx_padded_eval = [0] * pad_len_eval + ctx_ids_seguros_eval
            tensor_ctx = torch.tensor([ctx_padded_eval], dtype=torch.long)
            cli_t = torch.tensor([cliente2idx.get(int(cid), 0)], dtype=torch.long)
            mes_t = torch.tensor([int(mes_actual)], dtype=torch.long)

            with torch.no_grad():
                out_eval, attn_eval = modelo_gru(tensor_ctx, cli_t, mes_t)
                scores_eval = torch.sigmoid(out_eval[0]).numpy()
                pesos_attn_eval = attn_eval[0].squeeze(-1).numpy()

            indices_ordenados = scores_eval.argsort()[::-1]
            top_5_filtrado = []
            for idx_prod in indices_ordenados:
                if idx_prod == 0: continue
                if len(top_5_filtrado) >= 5: break
                nombre_prod_eval = mapa_productos.get(idx_prod, "")
                es_seguro_eval, _ = pasa_filtros_seguridad(nombre_prod_eval, contexto_nombres, zona_activa, compras_ctx, mapa_productos)
                if es_seguro_eval:
                    top_5_filtrado.append(idx_prod)

            productos_sugeridos_unicos.update(top_5_filtrado)
            hr_total += 1 if ground_truth_id in top_5_filtrado[:5] else 0
            if ground_truth_id in top_5_filtrado[:5]:
                idx_gt = top_5_filtrado.index(ground_truth_id)
                ndcg_total += 1 / np.log2(idx_gt + 2)
            atencion_media += np.max(pesos_attn_eval)
            casos_validos += 1

        if casos_validos > 0:
            hr_final = (hr_total / casos_validos) * 100
            ndcg_final = ndcg_total / casos_validos
            atencion_final = (atencion_media / casos_validos) * 100
            cobertura_catalogo = (len(productos_sugeridos_unicos) / num_items) * 100
        else:
            hr_final = ndcg_final = atencion_final = cobertura_catalogo = 0

        return {
            "hit_rate_5": f"{hr_final:.1f}%",
            "ndcg_5": f"{ndcg_final:.3f}",
            "pico_atencion": f"{atencion_final:.1f}%",
            "cobertura_catalogo": f"{cobertura_catalogo:.1f}%"
        }
    except Exception as e:
        sys.stderr.write(f"[WARNING] Error al calcular telemetria zonal: {e}\n")
        return {
            "hit_rate_5": "0.0%",
            "ndcg_5": "0.000",
            "pico_atencion": "0.0%",
            "cobertura_catalogo": "0.0%"
        }


#### 10.4. Pipeline de Inferencia Zonal y Proyecciones de 3 Meses

In [ ]:
def ejecutar_inferencia_local(cliente_id, compras_ctx):
    col_zona = next((c for c in ['vendedor', 'zona'] if c in compras_ctx.columns), 'zona')
    historial_df = compras_ctx[compras_ctx['cliente_id'] == cliente_id]
    if historial_df.empty:
        return {"error": f"No se encontró historial para el cliente {cliente_id}"}

    zona_activa = historial_df[col_zona].iloc[0]
    num_items = int(compras_ctx['producto_id'].max()) + 2
    mapa_productos = compras_ctx.drop_duplicates('producto_id').set_index('producto_id')['producto'].to_dict()

    # Cargar motor de contenido
    motor_contenido = None
    try:
        motor_contenido = MotorContenido(compras_ctx, JSON_PATH)
        if motor_contenido.hay_productos_nuevos():
            max_id_actual = max(mapa_productos.keys()) if mapa_productos else 0
            for i, prod_nuevo in enumerate(motor_contenido.productos_nuevos()):
                if prod_nuevo not in mapa_productos.values():
                    mapa_productos[max_id_actual + 1 + i] = prod_nuevo
    except Exception as e:
        sys.stderr.write(f"[WARNING] Error MotorContenido: {e}\n")

    # Cargar modelo PyTorch
    clientes_unicos = sorted(compras_ctx['cliente_id'].unique().tolist())
    cliente2idx_fallback = {c: i + 1 for i, c in enumerate(clientes_unicos)}
    num_clientes_fallback = len(clientes_unicos) + 1

    modelo_gru, cliente2idx, num_clientes = cargar_modelo_gru_checkpoint(
        MODEL_PATH, num_items, num_clientes_fallback, cliente2idx_fallback
    )

    historial_ids = historial_df['producto_id'].tolist()
    historial_nombres = [mapa_productos[pid] for pid in historial_ids if pid in mapa_productos]

    es_cold_start = len(historial_ids) < 3
    cli_idx_tensor = torch.tensor([cliente2idx.get(cliente_id, 0)], dtype=torch.long)

    horizonte_meses = ["Mes Actual (En Curso)", "Mes +1 (Próximo Mes)", "Mes +2 (Proyección)"]
    proyecciones = {}
    historial_simulado = list(historial_nombres)
    historial_ids_simulado = list(historial_ids)

    mes_actual = int(pd.Timestamp.now().month)

    # Calcular telemetría zonal
    telemetria = calcular_telemetria_zonal(
        cliente_id, zona_activa, compras_ctx, modelo_gru, cliente2idx, mapa_productos, num_items
    )

    # Proyecciones por mes
    for paso, mes_nombre in enumerate(horizonte_meses):
        mes_prediccion = ((mes_actual + paso - 1) % 12) + 1
        mes_tensor = torch.tensor([mes_prediccion], dtype=torch.long)
        if es_cold_start:
            df_zona = compras_ctx[compras_ctx[col_zona] == zona_activa]
            top_ids_cs = df_zona[~df_zona['producto_id'].isin(historial_ids_simulado)].groupby('producto_id')['producto_id'].count().sort_values(ascending=False).head(5).index.tolist()
            candidatos = [(pid, 'Cold Start', 0.5) for pid in top_ids_cs if pid in mapa_productos]
            item_foco_nombre = "Popularidad de Zona"
            peso_max_pct = 100
        else:
            ctx_ids = historial_ids_simulado[-10:]
            max_emb_id = modelo_gru.item_embedding.num_embeddings - 1
            ctx_ids_seguros = [pid if pid <= max_emb_id else 0 for pid in ctx_ids]
            pad_len = max(0, 10 - len(ctx_ids_seguros))
            ctx_padded = [0] * pad_len + ctx_ids_seguros
            hist_tensor = torch.tensor([ctx_padded], dtype=torch.long)

            with torch.no_grad():
                logits, attn_weights = modelo_gru(hist_tensor, cli_idx_tensor, mes_tensor)
                scores_gru = torch.sigmoid(logits[0]).numpy()
                pesos_attn = attn_weights[0].squeeze(-1).numpy()

            idx_max_attn = np.argmax(pesos_attn)
            peso_max_pct = round(float(pesos_attn[idx_max_attn]) * 100, 1)
            item_foco_id = hist_tensor[0][idx_max_attn].item()
            item_foco_nombre = mapa_productos.get(item_foco_id, historial_simulado[-1] if historial_simulado else "Historial Base")

            # Obtener candidatos NCF
            candidatos_ncf = obtener_candidatos_ncf(
                cliente_id, cli_idx_tensor, modelo_gru, cliente2idx, compras_ctx, num_items, mapa_productos
            )

            top_indices_gru = scores_gru.argsort()[::-1][:10]
            candidatos = [(int(i), 'Atención-GRU', scores_gru[i]) for i in top_indices_gru if i > 0 and i in mapa_productos] + candidatos_ncf

        if motor_contenido and motor_contenido.hay_productos_nuevos() and not es_cold_start:
            mapa_inv = {v.upper(): k for k, v in mapa_productos.items()}
            candidatos_nuevos = inyectar_candidatos_nuevos(motor_contenido, historial_simulado, mapa_inv)
            for prod_id_n, motor_n, score_n, _ in candidatos_nuevos:
                candidatos.append((prod_id_n, motor_n, score_n))
        candidatos.sort(key=lambda x: x[2], reverse=True)

        recomendaciones_mes = []
        aprobadas = 0
        for prod_id, motor, score_raw in candidatos:
            if aprobadas >= 3: break
            if isinstance(prod_id, str) and prod_id.startswith("NUEVO_"):
                nombre_prod = prod_id.replace("NUEVO_", "")
            else:
                nombre_prod = mapa_productos.get(prod_id, str(prod_id))
            es_seguro, _ = pasa_filtros_seguridad(nombre_prod, historial_simulado, zona_activa, compras_ctx, mapa_productos)
            if not es_seguro: continue
            es_autoreg = (paso > 0)
            explicacion = generar_explicacion(nombre_prod, historial_simulado, motor, mes_nombre, item_foco=item_foco_nombre, peso=peso_max_pct if motor == 'Atención-GRU' else 100, es_autorregresivo=es_autoreg)
            recomendaciones_mes.append({
                "producto": nombre_prod,
                "motor_origen": motor,
                "explicacion": explicacion
            })
            aprobadas += 1
            if aprobadas == 1:
                historial_simulado.append(nombre_prod)
                historial_ids_simulado.append(prod_id)
        proyecciones[mes_nombre] = recomendaciones_mes

    return {
        "zona": zona_activa,
        "telemetria": telemetria,
        "proyecciones": proyecciones
    }


#### 10.5. Ejecución de Predicción y Reporte

In [ ]:
resultado_prediccion = ejecutar_inferencia_local(CLIENTE_ID_PRUEBA, df_compras)

if "error" in resultado_prediccion:
    print(f"Error al procesar predicción: {resultado_prediccion['error']}")
else:
    telemetria_zonal = resultado_prediccion.get("telemetria", {})
    print(f"--- TELEMETRÍA LOCAL DEL CLIENTE (Zona: {resultado_prediccion.get('zona', 'N/A')}) ---")
    print(f"Hit Rate @ 5 Local:     {telemetria_zonal.get('hit_rate_5', '0.0%')}")
    print(f"NDCG @ 5 Local:         {telemetria_zonal.get('ndcg_5', '0.000')}")
    print(f"Pico de Atención:       {telemetria_zonal.get('pico_atencion', '0.0%')}")
    print(f"Cobertura de Catálogo:  {telemetria_zonal.get('cobertura_catalogo', '0.0%')}\n")

    for mes, recomendaciones in resultado_prediccion.get("proyecciones", {}).items():
        print(f"--- {mes.upper()} ---")
        for idx, rec in enumerate(recomendaciones, 1):
            print(f"  {idx}. Producto:    {rec.get('producto', 'N/A')}")
            print(f"     Algoritmo:   {rec.get('motor_origen', 'N/A')}")
            print(f"     Explicación: {rec.get('explicacion', 'N/A')}")
        print("-" * 50)

--- TELEMETRÍA LOCAL DEL CLIENTE (Zona: PHARMA - N2) ---
Hit Rate @ 5 Local:     81.2%
NDCG @ 5 Local:         0.658
Pico de Atención:       86.0%
Cobertura de Catálogo:  88.2%

--- MES ACTUAL (EN CURSO) ---
  1. Producto:    LANDAX
     Algoritmo:   NCF
     Explicación: Oportunidad Cross-Selling: Clinicas con perfil de compra similar al tuyo que adquieren ZEBESTEN tambien consumen LANDAX.
  2. Producto:    LAGRICEL PF
     Algoritmo:   Atención-GRU
     Explicación: Reposición Sugerida: Ciclo de compra detecta demanda inminente para LAGRICEL PF basado en consumo de ZEBESTEN (99.3% relevancia).
  3. Producto:    ZEBESTEN
     Algoritmo:   Atención-GRU
     Explicación: Reposición Sugerida: Ciclo de compra detecta demanda inminente para ZEBESTEN basado en consumo de ZEBESTEN (99.3% relevancia).
--------------------------------------------------
--- MES +1 (PRÓXIMO MES) ---
  1. Producto:    SOPHIPREN
     Algoritmo:   Atención-GRU
     Explicación: Reposición Sugerida: Ciclo de compra 

## 11. Diagnóstico de Nuevos Lanzamientos (TF-IDF vs Portafolio del Cliente)

In [ ]:
if not JSON_PATH.exists():
    print(f"El archivo de metadatos de productos nuevos no existe en {JSON_PATH}")
else:
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        prod_metadata = json.load(f)

    if not prod_metadata:
        print("No hay productos nuevos registrados para analizar.")
    else:
        motor = MotorContenido(df_compras, JSON_PATH)
        productos_cliente = set(historial_cliente['producto'].unique().tolist())

        for prod_nuevo in prod_metadata.keys():
            print(f"NUEVO LANZAMIENTO: {prod_nuevo}")
            print(f"Composición: {prod_metadata[prod_nuevo].get('composicion', '')}")
            print(f"Indicación:  {prod_metadata[prod_nuevo].get('indicacion', '')}\n")

            df_sim = motor.tabla_similitud_producto_nuevo(prod_nuevo)
            if df_sim.empty:
                print("No se encontraron similitudes.")
                continue

            df_sim_cliente = df_sim[df_sim['Producto existente'].isin(productos_cliente)].copy()
            if df_sim_cliente.empty:
                print("El cliente no consume productos clínicamente afines a este lanzamiento.\n")
            else:
                columnas_consola = ['Producto existente', 'Sub-familia', 'Similitud TF-IDF', '¿Por qué?']
                display(df_sim_cliente[columnas_consola])
            print("=" * 60)

NUEVO LANZAMIENTO: SPLASH TEARS
Composición: Condroit[in Sulfato de Sodio, Hipromelosa
Indicación:  Aivio del ojo seco



,Producto existente,Sub-familia,Similitud TF-IDF,¿Por qué?
1,LAGRICEL PF,Lubricante ocular sin conservantes,18.1%,"comparten: hidratante, lubricante, lagrimal"
3,ELIPTIC PF,Lubricante ocular sin conservantes,11.6%,"comparten: hidratante, lubricante, lagrimal"
5,GAAP PF,Lubricante ocular sin conservantes,11.5%,"comparten: hidratante, lubricante, lagrimal"
6,AQUADRAN,Lubricante ocular gel,7.5%,"comparten: hidratante, lubricante, lagrimal"
7,DUSTALOX,Antibiótico antiinflamatorio ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"
8,FLUMETOL NF,Corticoide ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"
9,SOPHIPREN,Corticoide ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"
10,ELAR-B,Corticoide antibiótico ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"
13,ZEBESTEN,Antihistamínico antialérgico ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"
14,AGGLAD,Antihistamínico antialérgico ocular,0.3%,"mismo formato (Gotas) | comparten: ocular, gotas"


NUEVO LANZAMIENTO: SPLASH TEARS PF
Composición: Hipromelosa
Indicación:  Alivio del ojo seco



,Producto existente,Sub-familia,Similitud TF-IDF,¿Por qué?
3,ELIPTIC PF,Lubricante ocular sin conservantes,13.2%,"comparten: hidratante, lubricante, lagrimal"
5,GAAP PF,Lubricante ocular sin conservantes,13.1%,"comparten: hidratante, lubricante, lagrimal"
1,LAGRICEL PF,Lubricante ocular sin conservantes,11.9%,"comparten: hidratante, lubricante, lagrimal"
6,AQUADRAN,Lubricante ocular gel,10.5%,"comparten: hidratante, lubricante, lagrimal"
7,DUSTALOX,Antibiótico antiinflamatorio ocular,0.0%,comparten: ocular
8,FLUMETOL NF,Corticoide ocular,0.0%,comparten: ocular
10,ELAR-B,Corticoide antibiótico ocular,0.0%,comparten: ocular
9,SOPHIPREN,Corticoide ocular,0.0%,comparten: ocular
13,ZEBESTEN,Antihistamínico antialérgico ocular,0.0%,comparten: ocular
14,AGGLAD,Antihistamínico antialérgico ocular,0.0%,comparten: ocular
